# Chest X-Ray Pneumonia Detection: Deep Technical Exploration
### End-to-End Pipeline: Data Leakage Prevention, CLAHE Preprocessing, DenseNet-121, Grad-CAM, & Multimodal XGBoost Fusion

This notebook demonstrates the complete clinical workflow for AI-assisted thoracic radiography triage:
1. **Pathology & Clinical Problem Formulation**
2. **Patient-Level Leakage Mitigation & Stratified Splitting**
3. **Contrast-Limited Adaptive Histogram Equalization (CLAHE)**
4. **Convolutional Architectures & Two-Phase Fine-Tuning**
5. **Handling Severe Class Imbalance (Class-Weighted Loss & Focal Loss)**
6. **Clinical Evaluation Metrics (Sensitivity/Recall, Specificity, ROC-AUC, ECE Calibration)**
7. **Visual Explainability (Grad-CAM) & Shortcut Learning Audits**
8. **Multimodal Fusion: Combining Imaging Probabilities with Patient Vital Signs**

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

# Add root project to sys.path
sys.path.append(os.path.abspath(".."))

from src.preprocessing import apply_clahe, preprocess_xray_image, denormalize_image
from src.dataset import extract_patient_id, patient_stratified_split, CLASS_NAMES_4
from src.models import get_model, DenseNet121Classifier, ResNet50Classifier, EfficientNetB0Classifier
from src.evaluate import calculate_clinical_metrics, tune_decision_threshold, plot_confusion_matrix_figure
from src.ensemble import SoftVotingEnsemble, MultimodalFusionModel
from src.explain import GradCAM, ClinicalTabularExplainer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Patient-Level Data Leakage Demonstration
In chest radiograph datasets (e.g., Kermany, CheXpert, NIH ChestX-ray14), multiple images frequently originate from the **same patient** taken at different intervals. 
Filenames often contain the patient ID (e.g. `person123_bacteria_456.jpeg`).

> **Critical Mistake:** Splitting data by image causes identical patient anatomies to appear in both training and test sets, artificially inflating accuracy by 5-15% while failing to generalize clinically.

In [ ]:
# Example filenames demonstrating patient ID extraction
sample_files = [
    "person100_bacteria_475.jpeg",
    "person100_bacteria_476.jpeg",
    "person100_bacteria_477.jpeg",
    "NORMAL2-IM-0315-0001.jpeg",
    "COVID-19 (104).png",
    "patient0023_study1_view1.png"
]

for f in sample_files:
    p_id = extract_patient_id(f)
    print(f"{f:32s} -> Extracted Patient ID: {p_id}")

## 2. CLAHE Contrast Equalization Comparison
Chest X-rays often suffer from improper penetration or scanner exposure variations. Contrast-Limited Adaptive Histogram Equalization (**CLAHE**) operates on local grid tiles ($8 \times 8$) and applies clip limits ($2.0$) to avoid amplifying high-frequency sensor noise while sharply elucidating subtle parenchymal infiltrates.

In [ ]:
# Load a benchmark study
bench_path = "../data/samples/patient_0204_bacteria.png"
if os.path.exists(bench_path):
    raw = cv2.imread(bench_path, cv2.IMREAD_GRAYSCALE)
    enhanced = apply_clahe(raw, clip_limit=2.0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), dpi=120)
    ax1.imshow(raw, cmap="gray")
    ax1.set_title("Original Radiograph", fontsize=12, weight="bold")
    ax1.axis("off")

    ax2.imshow(enhanced, cmap="gray")
    ax2.set_title("CLAHE Enhanced (Clip Limit 2.0)", fontsize=12, weight="bold")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()

## 3. Deep Learning Architectures & Feature Extraction
We compare four architectures:
- **DenseNet-121:** Feature reuse through dense connectivity ($1024$-d feature embeddings)
- **ResNet-50:** Residual identity shortcuts solving vanishing gradients ($2048$-d embeddings)
- **EfficientNet-B0:** Compound scaling of network depth, width, and resolution ($1280$-d embeddings)
- **Custom CNN:** 4-Block baseline for foundational benchmarking

In [ ]:
# Test forward passes and feature extraction dimensions
dummy_input = torch.randn(2, 3, 224, 224)

models_dict = {
    "DenseNet-121": get_model("densenet121", num_classes=4, pretrained=False),
    "ResNet-50": get_model("resnet50", num_classes=4, pretrained=False),
    "EfficientNet-B0": get_model("efficientnet_b0", num_classes=4, pretrained=False),
    "Custom CNN": get_model("custom_cnn", num_classes=4)
}

for name, m in models_dict.items():
    m.eval()
    logits = m(dummy_input)
    features = m.extract_features(dummy_input)
    print(f"{name:16s} | Logits: {list(logits.shape)} | Feature Vector Dimension: {features.shape[1]}")

## 4. Visual Saliency via Grad-CAM & Shortcut Learning Verification
Grad-CAM computes the gradients of the target class score with respect to the feature activation maps of the final convolutional layer:
$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{i,j}^k}$$
$$L_{\text{Grad-CAM}}^c = \text{ReLU}\left( \sum_k \alpha_k^c A^k \right)$$

We also calculate the **Parenchymal Concentration Ratio** to ensure the model attends to anatomical lung pathology rather than radiologist text markers or hospital stamps.

In [ ]:
dense_model = models_dict["DenseNet-121"]
gradcam = GradCAM(dense_model)

heatmap = gradcam.generate_heatmap(dummy_input[0:1], target_class=1)
audit = gradcam.check_shortcut_learning(heatmap)

print("Grad-CAM Saliency Map shape:", heatmap.shape)
print("Shortcut Learning Audit:", audit)

## 5. Multimodal Clinical Fusion (XGBoost)
Combines image probabilities with patient vitals (`SpO2`, `Temperature`, `Heart Rate`, `Respiratory Rate`, `Age`) to produce a calibrated risk score.

In [ ]:
fusion = MultimodalFusionModel()
fusion.fit_synthetic_benchmark(num_samples=1000)

# Simulating a severe pneumonia clinical presentation
cnn_probs = np.array([0.03, 0.85, 0.08, 0.04])
patient_vitals = {
    "age": 58,
    "sex": "Male",
    "temperature": 39.2,
    "spo2": 91.0,
    "respiratory_rate": 26,
    "heart_rate": 104,
    "cough": True,
    "fever": True
}

risk_result = fusion.predict_fusion_risk(cnn_probs, patient_vitals)
print("Multimodal Risk Triage Result:")
print(f"  Final Risk Score: {risk_result['final_risk_score']*100:.1f}%")
print(f"  Risk Stratification Tier: {risk_result['risk_tier']}")
print(f"  Actionable Guidance: {risk_result['clinical_recommendation']}")